In [ ]:
!pip install torch torchvision fastapi streamlit pandas matplotlib opencv-python pillow pyngrok kagglehub numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 52.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import os
import cv2
import numpy as np
from collections import Counter
import streamlit as st
from PIL import Image
from pyngrok import ngrok
from PIL import Image
import colorsys
from sklearn.cluster import KMeans


In [ ]:
music_path = kagglehub.dataset_download("joebeachcapital/30000-spotify-songs")
fashion_path = kagglehub.dataset_download("nguyngiabol/colorful-fashion-dataset-for-object-detection")

Using Colab cache for faster access to the '30000-spotify-songs' dataset.


100%|██████████| 104M/104M [00:01<00:00, 98.5MB/s] 

Extracting files...


#Data Preparation

In [ ]:
music_files = [f for f in os.listdir(music_path) if f.endswith('.csv')]
if music_files:
    df_music = pd.read_csv(os.path.join(music_path, music_files[0]))

In [ ]:
fashion_files = os.listdir(fashion_path)
csv_files = [f for f in fashion_files if f.endswith('.csv')]
image_files = [f for f in fashion_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if csv_files:
    df_fashion = pd.read_csv(os.path.join(fashion_path, csv_files[0]))
else:
    image_paths = [os.path.join(fashion_path, f) for f in image_files]

# Clothing

In [ ]:
def get_color_name(rgb):
    r, g, b = [x / 255.0 for x in rgb]
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    h = h * 360

    if v < 0.15:
        return "black"
    if v > 0.85 and s < 0.25:
        return "white"
    if s < 0.2:
        return "gray"

    if (h >= 0 and h < 15) or (h >= 345 and h <= 360):
        return "red"
    elif h >= 15 and h < 45:
        return "orange"
    elif h >= 45 and h < 70:
        return "yellow"
    elif h >= 70 and h < 165:
        return "green"
    elif h >= 165 and h < 250:
        return "blue"
    elif h >= 250 and h < 290:
        return "purple"
    elif h >= 290 and h < 330:
        return "pink"
    elif h >= 20 and h < 45 and v < 0.6:
        return "brown"
    else:
        return "other"

In [ ]:
def color_detection(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    colors = []

    for i in range(0, width, 20):
        for j in range(0, height, 20):
            pixel = image[j, i]
            colors.append(get_color_name(pixel))

    return Counter(colors).most_common(1)[0][0]

In [ ]:
def remove_skin_and_background(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 30, 60], dtype=np.uint8)
    upper_skin = np.array([35, 255, 255], dtype=np.uint8)
    skin_mask = cv2.inRange(image_hsv, lower_skin, upper_skin)

    height, width = image.shape[:2]
    border_size = int(min(height, width) * 0.05)
    background_mask = np.zeros((height, width), dtype=np.uint8)
    background_mask[:border_size, :] = 255
    background_mask[-border_size:, :] = 255
    background_mask[:, :border_size] = 255
    background_mask[:, -border_size:] = 255

    combined_mask = cv2.bitwise_or(skin_mask, background_mask)
    clothing_mask = cv2.bitwise_not(combined_mask)

    kernel = np.ones((5, 5), np.uint8)
    clothing_mask = cv2.morphologyEx(clothing_mask, cv2.MORPH_OPEN, kernel)
    clothing_mask = cv2.morphologyEx(clothing_mask, cv2.MORPH_CLOSE, kernel)

    clothing_only = cv2.bitwise_and(image_rgb, image_rgb, mask=clothing_mask)
    return clothing_only, clothing_mask


In [ ]:
def load_image(image_path):
    img = Image.open(image_path).convert("RGB")
    return np.array(img)

# Music

In [ ]:
df_music.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [ ]:
df_music["playlist_genre"].unique()

array(['pop', 'rap', 'rock', 'latin', 'r&b', 'edm'], dtype=object)

In [ ]:
def recommend_music_by_color(color):
    color_music_map = {
        'red': {'valence': 0.8, 'energy': 0.9, 'genre': 'rock'},
        'blue': {'valence': 0.5, 'energy': 0.4, 'genre': 'r&b'},
        'black': {'valence': 0.3, 'energy': 0.7, 'genre': 'edm'},
        'white': {'valence': 0.7, 'energy': 0.5, 'genre': 'pop'},
        'pink': {'valence': 0.9, 'energy': 0.6, 'genre': 'pop'},
        'green': {'valence': 0.6, 'energy': 0.5, 'genre': 'latin'},
        'purple': {'valence': 0.7, 'energy': 0.8, 'genre': 'r&b'},
        'orange': {'valence': 0.8, 'energy': 0.7, 'genre': 'rap'},
        'yellow': {'valence': 0.9, 'energy': 0.8, 'genre': 'pop'},
        'light_red': {'valence': 0.7, 'energy': 0.8, 'genre': 'pop'},
        'light_blue': {'valence': 0.6, 'energy': 0.5, 'genre': 'r&b'},
        'light_green': {'valence': 0.7, 'energy': 0.6, 'genre': 'latin'},
        'light_pink': {'valence': 0.8, 'energy': 0.5, 'genre': 'pop'},
        'light_purple': {'valence': 0.6, 'energy': 0.7, 'genre': 'edm'},
        'brown': {'valence': 0.5, 'energy': 0.4, 'genre': 'rap'},
        'light_brown': {'valence': 0.6, 'energy': 0.5, 'genre': 'r&b'},
        'gray': {'valence': 0.4, 'energy': 0.5, 'genre': 'edm'},
        'other': {'valence': 0.6, 'energy': 0.6, 'genre': 'pop'}
    }

    music_params = color_music_map.get(color, {'valence': 0.6, 'energy': 0.6})
    recommended_songs = df_music[
        (df_music['valence'] > music_params['valence'] - 0.1) &
        (df_music['valence'] < music_params['valence'] + 0.1) &
        (df_music['energy'] > music_params['energy'] - 0.1) &
        (df_music['energy'] < music_params['energy'] + 0.1)
    ].head(5)
    return recommended_songs

# Streamlit


In [ ]:
%%writefile app.py

def color_detection(image_path):
    clothing_only, mask = remove_skin_and_background(image_path)

    colors = []
    height, width = clothing_only.shape[:2]

    for i in range(0, width, 10):
        for j in range(0, height, 10):
            if mask[j, i] > 0:
                pixel = clothing_only[j, i]
                if np.mean(pixel) > 15:
                    colors.append(get_color_name(pixel))

    if not colors:
        return "other"

    return Counter(colors).most_common(1)[0][0]

Writing app.py


In [ ]:
st.title("Fashion Music Recommender")
uploaded_file = st.file_uploader("Upload your outfit photo", type=['jpg', 'jpeg', 'png'])

if uploaded_file is not None:
    with open("temp_image.jpg", "wb") as f:
        f.write(uploaded_file.getbuffer())

    image = Image.open(uploaded_file)
    st.image(image, caption="Your Outfit", use_column_width=True)

    color = color_detection("temp_image.jpg")
    st.write(f"Detected Color: **{color}**")

    recommended_songs = recommend_music_by_color(color)
    st.write("Recommended Songs:")

    for idx, song in recommended_songs.iterrows():
        st.write(f"- **{song['track_name']}** by {song['track_artist	']}")
        st.write(f"  Valence: {song['valence']:.2f} | Energy: {song['energy']:.2f}")

2026-03-06 23:31:39.350 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 23:31:39.439 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-06 23:31:39.441 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 23:31:39.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 23:31:39.445 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 23:31:39.446 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 23:31:39.449 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 23:31:39.451 Thread 'MainThread': mi

#Demonstration

In [ ]:
print("Testing")

if 'image_paths' in locals() and len(image_paths) > 0:
    for i, test_image in enumerate(image_paths[:3]):
        print(f"\nTest {i+1}: {test_image.split('/')[-1]}")

        color = color_detection(test_image)
        print(f"Detected Color (no skin/background): {color}")

        songs = recommend_music_by_color(color)
        print(f"Songs: {len(songs)}")

        for idx, song in songs.iterrows():
            print(f"- {song['track_name']} by {song['track_artist']}")
            print(f"  V: {song['valence']:.2f} | E: {song['energy']:.2f}")

else:
    print("No images found")

    from google.colab import files
    uploaded = files.upload()

    if uploaded:
        for filename in uploaded.keys():
            print(f"Testing: {filename}")
            color = color_detection(filename)
            print(f"Detected Color (no skin/background): {color}")

            songs = recommend_music_by_color(color)
            print(f"Songs: {len(songs)}")

            for idx, song in songs.iterrows():
                print(f"- {song['track_name']} by {song['track_artist']}")

Testing
No images found


Saving green.png to green.png
Testing: green.png
Detected Color (no skin/background): green
Songs: 5
- bad guy (with Justin Bieber) by Billie Eilish
- Closer (feat. Halsey) by The Chainsmokers
- It Ain’t Me (with Selena Gomez) by Kygo
- No Goodbye by Paul Kalkbrenner
- Lose Control by MEDUZA
